# Projeto de assistente de pedido do Mc Donalds
Um agente conversacional em português que ajuda a montar um pedido do McDonald's. Ele entende restrições escritas livremente ("frango com baixo sódio"). Se o usuário não disser nenhuma, o agente pergunta. As respostas vêm do `menu.csv`, não da memória do modelo.

### Instalações


In [14]:
!pip install langchain langchain_openai nltk

### Chave e modelo

In [15]:
from getpass import getpass

OPENAI_API_KEY = getpass()

··········


In [16]:
from langchain_openai import ChatOpenAI
modelo = ChatOpenAI(model="gpt-6-luna", api_key=OPENAI_API_KEY)

print(modelo.invoke("Responda apenas 'ok' se estiver funcionando.").content)

ok


### Carregar o corpus
Baixa o csv do GitHub

In [17]:
import requests
import pandas as pd
from io import StringIO

URL_MENU = "https://raw.githubusercontent.com/BeatrizGiacon/Projeto-Pratico-PLN-/main/menu.csv"

resposta = requests.get(URL_MENU, timeout=30)
resposta.raise_for_status()
df_bruto = pd.read_csv(StringIO(resposta.text))

print(df_bruto.shape)
df_bruto.head()


(260, 24)


,Category,Item,Serving Size,Calories,Calories from Fat,Total Fat,Total Fat (% Daily Value),Saturated Fat,Saturated Fat (% Daily Value),Trans Fat,...,Carbohydrates,Carbohydrates (% Daily Value),Dietary Fiber,Dietary Fiber (% Daily Value),Sugars,Protein,Vitamin A (% Daily Value),Vitamin C (% Daily Value),Calcium (% Daily Value),Iron (% Daily Value)
0,Breakfast,Egg McMuffin,4.8 oz (136 g),300,120,13.0,20,5.0,25,0.0,...,31,10,4,17,3,17,10,0,25,15
1,Breakfast,Egg White Delight,4.8 oz (135 g),250,70,8.0,12,3.0,15,0.0,...,30,10,4,17,3,18,6,0,25,8
2,Breakfast,Sausage McMuffin,3.9 oz (111 g),370,200,23.0,35,8.0,42,0.0,...,29,10,4,17,2,14,8,0,25,10
3,Breakfast,Sausage McMuffin with Egg,5.7 oz (161 g),450,250,28.0,43,10.0,52,0.0,...,30,10,4,17,2,21,15,0,30,15
4,Breakfast,Sausage McMuffin with Egg Whites,5.7 oz (161 g),400,210,23.0,35,8.0,42,0.0,...,30,10,4,17,2,21,6,0,25,10


In [18]:
renomear = {
    "Category": "categoria", "Item": "item", "Serving Size": "porcao_texto",
    "Calories": "calorias", "Total Fat": "gordura_g", "Saturated Fat": "gordura_sat_g",
    "Cholesterol": "colesterol_mg", "Sodium": "sodio_mg", "Carbohydrates": "carbo_g",
    "Sugars": "acucar_g", "Protein": "proteina_g", "Calcium (% Daily Value)": "calcio_vd_pct",
}
df = df_bruto[list(renomear)].rename(columns=renomear).copy()

# Limpa símbolos e espaços dos textos
df["item"] = df["item"].str.replace(r"[®™]", "", regex=True).str.strip()
df["categoria"] = df["categoria"].str.strip()

# Peso da porção em gramas (bebidas medidas em ml ficam sem valor)
df["porcao_g"] = df["porcao_texto"].str.extract(r"\((\d+(?:\.\d+)?) g\)").astype(float)

print("Itens com nome repetido:", df["item"].duplicated().sum())
df.head()

Itens com nome repetido: 0


,categoria,item,porcao_texto,calorias,gordura_g,gordura_sat_g,colesterol_mg,sodio_mg,carbo_g,acucar_g,proteina_g,calcio_vd_pct,porcao_g
0,Breakfast,Egg McMuffin,4.8 oz (136 g),300,13.0,5.0,260,750,31,3,17,25,136.0
1,Breakfast,Egg White Delight,4.8 oz (135 g),250,8.0,3.0,25,770,30,3,18,25,135.0
2,Breakfast,Sausage McMuffin,3.9 oz (111 g),370,23.0,8.0,45,780,29,2,14,25,111.0
3,Breakfast,Sausage McMuffin with Egg,5.7 oz (161 g),450,28.0,10.0,285,860,30,2,21,30,161.0
4,Breakfast,Sausage McMuffin with Egg Whites,5.7 oz (161 g),400,23.0,8.0,50,880,30,2,21,25,161.0


### Preparar o texto do menu

Os nomes dos itens ("Big Mac", "Chicken McNuggets") são o único texto do corpus. Este passo segue o tratamento de:

- Normalização (minúsculas, sem acentos) e tokenização.
- Remover palavras vazias como "the" e "of" (stopwords) e faz lematização.
- Criar etiquetas úteis.


In [19]:
import re, unicodedata, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords"); nltk.download("wordnet"); nltk.download("omw-1.4")
stop_en = set(stopwords.words("english"))
lematizador = WordNetLemmatizer()

def preprocessar_nome(texto):
    """Normaliza, tokeniza, remove stopwords e lematiza um nome de item."""
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower()
    tokens = re.findall(r"[a-z]+", texto)
    return [lematizador.lemmatize(t) for t in tokens if t not in stop_en and len(t) > 2]

df["tokens"] = df["item"].apply(preprocessar_nome)
df[["item", "tokens"]].head(10)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,item,tokens
0,Egg McMuffin,"[egg, mcmuffin]"
1,Egg White Delight,"[egg, white, delight]"
2,Sausage McMuffin,"[sausage, mcmuffin]"
3,Sausage McMuffin with Egg,"[sausage, mcmuffin, egg]"
4,Sausage McMuffin with Egg Whites,"[sausage, mcmuffin, egg, white]"
5,Steak & Egg McMuffin,"[steak, egg, mcmuffin]"
6,"Bacon, Egg & Cheese Biscuit (Regular Biscuit)","[bacon, egg, cheese, biscuit, regular, biscuit]"
7,"Bacon, Egg & Cheese Biscuit (Large Biscuit)","[bacon, egg, cheese, biscuit, large, biscuit]"
8,"Bacon, Egg & Cheese Biscuit with Egg Whites (R...","[bacon, egg, cheese, biscuit, egg, white, regu..."
9,"Bacon, Egg & Cheese Biscuit with Egg Whites (L...","[bacon, egg, cheese, biscuit, egg, white, larg..."


### Documentação para busca semântica
Transforma cada linha em um Document do LangChain, com um texto descritivo e metadados. É a base do RAG.

In [20]:
from langchain_core.documents import Document

def descrever(linha):
    return (f"{linha.item} (categoria: {linha.categoria}). Calorias: {linha.calorias:.0f}; "
            f"sódio: {linha.sodio_mg:.0f} mg; gordura: {linha.gordura_g:.0f} g; "
            f"açúcar: {linha.acucar_g:.0f} g; proteína: {linha.proteina_g:.0f} g.")

documentos = [
    Document(page_content=descrever(l), metadata={"item": l.item, "categoria": l.categoria})
    for l in df.itertuples()
]

**Busca por similaridade de textos**

Usamos TF-IDF com similaridade de cosseno, sem depender de embeddings externos. A busca usa apenas linhas do CSV e é reproduzível. O ranking por relevância é separado da aplicação de restrições numéricas.

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

corpus_busca = [f"{d.metadata['item']} {d.metadata['categoria']} {d.page_content}" for d in documentos]
vetorizador = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2), stop_words="english")
matriz_tfidf = vetorizador.fit_transform(corpus_busca)

def buscar_similares(consulta: str, ids_permitidos=None, k=8):
    """Retorna IDs existentes no CSV, ordenados por similaridade cosseno."""
    ids = list(range(len(df))) if ids_permitidos is None else list(ids_permitidos)
    if not ids:
        return []
    consulta_vetor = vetorizador.transform([consulta])
    pontuacoes = cosine_similarity(consulta_vetor, matriz_tfidf[ids]).ravel()
    ordem = np.argsort(-pontuacoes, kind="stable")
    return [(int(ids[j]), float(pontuacoes[j])) for j in ordem[:k]]

print("Exemplo de busca:", [(df.iloc[i]['item'], round(s, 3)) for i, s in buscar_similares('chicken nuggets', k=5)])

Exemplo de busca: [('Premium McWrap Chicken & Ranch (Grilled Chicken)', 0.449), ('Premium McWrap Chicken & Bacon (Grilled Chicken)', 0.448), ('Premium McWrap Chicken & Ranch (Crispy Chicken)', 0.447), ('Premium McWrap Chicken & Bacon (Crispy Chicken)', 0.436), ('Premium McWrap Southwest Chicken (Crispy Chicken)', 0.432)]


2. Classificação de intenção e extração de restrições

O LLM recebe instruções para converter a frase livre em uma estrutura validada pelo Pydantic. A classificação identifica intenções como recomendar, adicionar, remover, visualizar pedido e perguntar. A extração reconhece preferências e limites nutricionais explicitamente mencionados, sem presumir que “baixo sódio” corresponda a um limite médico universal.

Para tornar termos como “baixo sódio” operacionais, usamos um ranking relativo (menor teor de sódio entre os candidatos), não uma certificação de produto com baixo sódio. Limites numéricos, como “até 500 mg”, são filtros rígidos

In [22]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Interpretacao(BaseModel):
    intencao: Literal["recomendar", "adicionar", "remover", "ver_pedido", "pergunta", "reiniciar"] = "recomendar"
    consulta: str = Field(description="Termos do produto procurado; preserve nomes e ingredientes mencionados")
    restricoes_informadas: bool = Field(description="True se o usuário expressou preferência ou restrição alimentar/nutricional")
    prefere_frango: bool = False
    evita_carne_bovina: bool = False
    evita_frango: bool = False
    prefere_menos_sodio: bool = False
    prefere_menos_calorias: bool = False
    prefere_menos_acucar: bool = False
    prefere_mais_proteina: bool = False
    max_sodio_mg: Optional[float] = None
    max_calorias: Optional[float] = None
    max_acucar_g: Optional[float] = None
    item_referido: Optional[str] = None
    pergunta: Optional[str] = None
    categoria_desejada: Optional[str] = None  # lanche, sobremesa, bebida, acompanhamento, cafe
    subtipo_lanche: Optional[str] = None  # frango, bovino, peixe, nuggets, outros
    alergias: list[str] = Field(default_factory=list)
    sem_alergias: bool = False
    confirmou_risco_alergia: bool = False

prompt_interpretacao = ChatPromptTemplate.from_messages([
    ("system", """Você interpreta mensagens em português para um assistente de cardápio.
Classifique a intenção e extraia somente preferências ou limites expressos pelo usuário.
Não invente produtos, ingredientes, alergênicos, números ou propriedades nutricionais.
'baixo sódio' = prefere_menos_sodio, SEM definir um limite numérico.
'frango' = prefere_frango; 'sem carne bovina' = evita_carne_bovina.
Se o usuário disser apenas 'quero um hambúrguer', restricoes_informadas=false.
Se disser 'sem restrições', restricoes_informadas=true.
Para adicionar/remover, identifique item_referido quando possível.
Extraia categoria_desejada e subtipo_lanche quando mencionados.
Extraia alergias declaradas (leite, ovo, trigo/glúten, soja, amendoim, castanhas, peixe, crustáceos, gergelim etc.).
NÃO infira alergia a partir de preferências ou de restrições nutricionais.
Se a pessoa disser explicitamente que não tem alergias, marque sem_alergias=true.
Se disser explicitamente que aceita apenas visualizar produtos sem garantia de segurança para alergias, marque confirmou_risco_alergia=true.
Perguntas sobre categorias/cardápio podem ter intencao='pergunta'; preserve a consulta.
A consulta deve reter palavras do usuário; não traduza automaticamente nomes comerciais."""),
    ("human", "{mensagem}")
])
interpretador = prompt_interpretacao | modelo.with_structured_output(Interpretacao)

def interpretar(mensagem):
    return interpretador.invoke({"mensagem": mensagem})

exemplo = interpretar("Quero frango com pouco sódio, até 500 mg")
print(exemplo.model_dump())

{'intencao': 'recomendar', 'consulta': 'frango', 'restricoes_informadas': True, 'prefere_frango': True, 'evita_carne_bovina': False, 'evita_frango': False, 'prefere_menos_sodio': True, 'prefere_menos_calorias': False, 'prefere_menos_acucar': False, 'prefere_mais_proteina': False, 'max_sodio_mg': 500.0, 'max_calorias': None, 'max_acucar_g': None, 'item_referido': None, 'pergunta': None, 'categoria_desejada': None, 'subtipo_lanche': None, 'alergias': [], 'sem_alergias': False, 'confirmou_risco_alergia': False}


3. Regras determinísticas sobre o CSV

A etapa abaixo filtra somente campos nutricionais presentes no CSV. Não existe lista de ingredientes nem de alergênicos neste arquivo; por isso, exclusões como “sem glúten”, “sem lactose” ou “sem cebola” não podem ser garantidas. A categoria/nome serve apenas para buscar candidatos, não para assegurar a composição.

In [23]:
# Verificação preventiva: os IDs precisam existir antes das recomendações.
# Se o CSV tiver sido recarregado, execute também a célula de preparação
# (seção "Carregar o corpus") e a de busca TF-IDF (seção 1).
colunas_necessarias = {"id_menu", "item", "categoria", "sodio_mg", "calorias", "acucar_g", "proteina_g"}
faltantes = colunas_necessarias - set(df.columns)
if faltantes == {"id_menu"}:
    df = df.reset_index(drop=True).copy()
    df["id_menu"] = range(len(df))
    print("Coluna id_menu recriada com sucesso.")
elif faltantes:
    raise ValueError(
        f"Colunas ausentes: {sorted(faltantes)}. "
        "Execute a célula de preparação do menu (renomear/df) antes desta seção."
    )
# As linhas precisam corresponder aos índices usados na matriz TF-IDF.
if len(df) != matriz_tfidf.shape[0] or df["id_menu"].tolist() != list(range(len(df))):
    raise ValueError("IDs e índice TF-IDF não estão alinhados. Reexecute a preparação do CSV e a seção 1 (TF-IDF).")

def normalizar(s):
    import unicodedata
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()

def filtrar_cardapio(p: Interpretacao):
    candidatos = df.copy()
    for coluna, limite in [("sodio_mg", p.max_sodio_mg), ("calorias", p.max_calorias), ("acucar_g", p.max_acucar_g)]:
        if limite is not None:
            if limite < 0:
                return candidatos.iloc[0:0]
            candidatos = candidatos[candidatos[coluna].notna() & (candidatos[coluna] <= limite)]
    nomes = candidatos["item"].map(normalizar)
    categorias = candidatos["categoria"].map(normalizar)
    # Heurística lexical: nomes de produtos NÃO comprovam ingredientes.
    if p.prefere_frango:
        candidatos = candidatos[nomes.str.contains(r"chicken|nugget|frango", regex=True)]
    if p.evita_frango:
        candidatos = candidatos[~candidatos["item"].map(normalizar).str.contains(r"chicken|nugget|frango", regex=True)]
    if p.evita_carne_bovina:
        # Não garantir ausência de carne bovina: só excluir nomes explicitamente associados.
        candidatos = candidatos[~candidatos["item"].map(normalizar).str.contains(r"beef|angus|quarter pounder", regex=True)]
    return candidatos

def recomendar_csv(p: Interpretacao, k=5):
    filtrados = filtrar_cardapio(p)
    if filtrados.empty:
        return []
    consulta = p.consulta or "menu"
    pares = buscar_similares(consulta, ids_permitidos=filtrados["id_menu"].tolist(), k=len(filtrados))
    pontuacao = dict(pares)
    registros = filtrados.copy()
    registros["similaridade"] = registros["id_menu"].map(pontuacao).fillna(0.0)
    # Preferências relativas são critérios de ordenação, não alegações clínicas.
    criterios = []
    asc = []
    for ativo, campo, crescente in [(p.prefere_menos_sodio,"sodio_mg",True), (p.prefere_menos_calorias,"calorias",True), (p.prefere_menos_acucar,"acucar_g",True), (p.prefere_mais_proteina,"proteina_g",False)]:
        if ativo:
            criterios.append(campo); asc.append(crescente)
    criterios += ["similaridade", "id_menu"]
    asc += [False, True]
    registros = registros.sort_values(criterios, ascending=asc, na_position="last")
    return registros.head(k).to_dict("records")

def formatar_item(r):
    def valor(chave, unidade):
        x = r.get(chave)
        return "não informado" if pd.isna(x) else f"{x:g} {unidade}"
    return (f"[ID {int(r['id_menu'])}] {r['item']} — {r['categoria']}; "
            f"{valor('calorias','kcal')}; sódio {valor('sodio_mg','mg')}; "
            f"açúcar {valor('acucar_g','g')}; proteína {valor('proteina_g','g')}")

print("Candidatos de exemplo:")
for item in recomendar_csv(exemplo):
    print(formatar_item(item))

Coluna id_menu recriada com sucesso.
Candidatos de exemplo:
[ID 87] Premium Southwest Salad (without Chicken) — Salads; 140 kcal; sódio 150 mg; açúcar 6 g; proteína 6 g
[ID 84] Premium Bacon Ranch Salad (without Chicken) — Salads; 140 kcal; sódio 300 mg; açúcar 4 g; proteína 9 g
[ID 78] Chicken McNuggets (4 piece) — Chicken & Fish; 190 kcal; sódio 360 mg; açúcar 0 g; proteína 9 g


Organização do cardápio em categorias e subtipos

As categorias originais do CSV são preservadas. A classificação abaixo cria grupos de navegação derivados somente de categoria e nome do produto, sem inferir ingredientes ou alergênicos. Produtos sem correspondência vão para outros. Os grupos são para navegação, não para comprovar composição.

In [24]:
import re

def grupo_produto(linha):
    cat = normalizar(linha['categoria'])
    nome = normalizar(linha['item'])
    if any(x in cat for x in ['dessert', 'sweet', 'sobremesa', 'shake', 'smoothie']):
        return 'sobremesa'
    if any(x in cat for x in ['beverage', 'drink', 'coffee', 'tea', 'bebida']):
        return 'bebida'
    if any(x in cat for x in ['breakfast', 'cafe da manha']):
        return 'cafe'
    if any(x in cat for x in ['side', 'salad', 'snack', 'acompanhamento']):
        return 'acompanhamento'
    if any(x in cat for x in ['beef', 'pork', 'chicken', 'fish', 'sandwich', 'burger', 'lanche']):
        return 'lanche'
    # Alguns CSVs colocam nuggets em categorias próprias.
    if any(x in nome for x in ['nugget', 'burger', 'sandwich', 'mcchicken', 'filet-o-fish']):
        return 'lanche'
    return 'outros'

def subtipo_produto(linha):
    if linha['grupo'] != 'lanche':
        return 'nao_aplicavel'
    nome = normalizar(linha['item'])
    cat = normalizar(linha['categoria'])
    if any(x in nome for x in ['nugget']): return 'nuggets'
    if any(x in nome for x in ['chicken', 'frango']) or 'chicken' in cat: return 'frango'
    if any(x in nome for x in ['fish', 'peixe']) or 'fish' in cat: return 'peixe'
    if any(x in nome for x in ['beef', 'angus', 'quarter pounder', 'big mac', 'hamburger', 'cheeseburger']) or 'beef' in cat: return 'bovino'
    return 'outros_lanches'

df['grupo'] = df.apply(grupo_produto, axis=1)
df['subtipo'] = df.apply(subtipo_produto, axis=1)
print('Categorias originais:', sorted(df['categoria'].unique().tolist()))
print('Grupos de navegação:')
display(df.groupby(['grupo', 'subtipo']).size().rename('quantidade').reset_index())

def mostrar_cardapio(grupo=None, subtipo=None, limite=12):
    itens = df
    if grupo: itens = itens[itens['grupo'] == grupo]
    if subtipo: itens = itens[itens['subtipo'] == subtipo]
    if itens.empty: return 'Nenhum produto encontrado nessa divisão do CSV.'
    linhas = [formatar_item(r) for r in itens.head(limite).to_dict('records')]
    resto = len(itens) - len(linhas)
    return '\n'.join(linhas) + (f'\n... e mais {resto} produtos. Refine o tipo ou peça mais opções.' if resto else '')

print('Exemplo de lanches de frango:')
print(mostrar_cardapio('lanche', 'frango', 5))

Categorias originais: ['Beef & Pork', 'Beverages', 'Breakfast', 'Chicken & Fish', 'Coffee & Tea', 'Desserts', 'Salads', 'Smoothies & Shakes', 'Snacks & Sides']
Grupos de navegação:


,grupo,subtipo,quantidade
0,acompanhamento,nao_aplicavel,19
1,bebida,nao_aplicavel,122
2,cafe,nao_aplicavel,42
3,lanche,bovino,15
4,lanche,frango,22
5,lanche,nuggets,5
6,sobremesa,nao_aplicavel,35


Exemplo de lanches de frango:
[ID 57] Premium Crispy Chicken Classic Sandwich — Chicken & Fish; 510 kcal; sódio 990 mg; açúcar 10 g; proteína 24 g
[ID 58] Premium Grilled Chicken Classic Sandwich — Chicken & Fish; 350 kcal; sódio 820 mg; açúcar 8 g; proteína 28 g
[ID 59] Premium Crispy Chicken Club Sandwich — Chicken & Fish; 670 kcal; sódio 1410 mg; açúcar 11 g; proteína 36 g
[ID 60] Premium Grilled Chicken Club Sandwich — Chicken & Fish; 510 kcal; sódio 1250 mg; açúcar 9 g; proteína 40 g
[ID 61] Premium Crispy Chicken Ranch BLT Sandwich — Chicken & Fish; 610 kcal; sódio 1400 mg; açúcar 11 g; proteína 32 g
... e mais 17 produtos. Refine o tipo ou peça mais opções.


4. Agente conversacional com ferramentas e RAG

O agente usa ferramentas LangChain para consultar os itens do CSV e o estado do pedido. A geração da resposta é fundamentada nos registros retornados pelas ferramentas; a seleção e a validação de IDs permanecem no código. Para evitar respostas sem suporte, a lista final de produtos é formatada deterministicamente, sem pedir ao LLM para reescrever valores nutricionais.

O estado mantém preferências entre turnos, permite adicionar/remover itens e pergunta sobre restrições na primeira solicitação quando elas não forem informadas.

In [25]:
from langchain_core.tools import tool

@tool
def consultar_menu(consulta: str) -> str:
    """Consulta produtos do menu.csv por similaridade textual, sem inventar itens."""
    resultados = buscar_similares(consulta, k=5)
    return "\n".join(formatar_item(df.iloc[i].to_dict()) for i, _ in resultados) or "Nenhum item encontrado."

@tool
def consultar_item_por_id(id_menu: int) -> str:
    """Obtém os valores nutricionais reais de um produto por ID no CSV."""
    if id_menu < 0 or id_menu >= len(df):
        return "ID inexistente no menu.csv."
    return formatar_item(df.iloc[id_menu].to_dict())

ferramentas = [consultar_menu, consultar_item_por_id]
print("Ferramentas disponíveis:", [t.name for t in ferramentas])

Ferramentas disponíveis: ['consultar_menu', 'consultar_item_por_id']


In [26]:
from dataclasses import dataclass, field

@dataclass
class EstadoPedido:
    preferencias: dict = field(default_factory=dict)
    pedido: list = field(default_factory=list)
    aguardando_restricoes: bool = False
    aguardando_categoria: bool = False
    consulta_pendente: str = ''
    categoria_atual: str = ''
    subtipo_atual: str = ''
    alergias: list = field(default_factory=list)
    sem_alergias_declaradas: bool = False
    ciente_limitacao_alergias: bool = False
    ultimas_recomendacoes: list = field(default_factory=list)
    historico: list = field(default_factory=list)

CAMPOS_PREF = ['prefere_frango','evita_carne_bovina','evita_frango','prefere_menos_sodio',
              'prefere_menos_calorias','prefere_menos_acucar','prefere_mais_proteina',
              'max_sodio_mg','max_calorias','max_acucar_g']
GRUPOS = {'lanche':'lanche', 'lanches':'lanche', 'hamburguer':'lanche',
          'sobremesa':'sobremesa', 'sobremesas':'sobremesa', 'doce':'sobremesa',
          'bebida':'bebida', 'bebidas':'bebida', 'acompanhamento':'acompanhamento',
          'acompanhamentos':'acompanhamento', 'cafe':'cafe', 'cafe da manha':'cafe'}
SUBTIPOS = {'frango':'frango','nuggets':'nuggets','nugget':'nuggets',
            'carne':'bovino','bovino':'bovino','hamburguer bovino':'bovino',
            'peixe':'peixe','outros':'outros_lanches'}

def extrair_grupo(texto):
    t = normalizar(texto)
    for palavra, grupo in sorted(GRUPOS.items(), key=lambda x: -len(x[0])):
        if re.search(r'(?<!\w)' + re.escape(palavra) + r'(?!\w)', t):
            return grupo
    return None

def extrair_subtipo(texto):
    t = normalizar(texto)
    for palavra, subtipo in sorted(SUBTIPOS.items(), key=lambda x: -len(x[0])):
        if re.search(r'(?<!\w)' + re.escape(palavra) + r'(?!\w)', t):
            return subtipo
    return None

AVISO_ALERGIA = ('ATENÇÃO: o CSV não informa alergênicos, ingredientes completos nem '
                'contaminação cruzada. Não é possível afirmar que qualquer item seja seguro. '
                'Consulte a tabela oficial atualizada e confirme com o restaurante antes de consumir.')

class AssistenteMcDonalds:
    def __init__(self):
        self.estado = EstadoPedido()

    def _preferencias(self, p):
        for campo in CAMPOS_PREF:
            valor = getattr(p, campo)
            if valor is not None and valor is not False:
                self.estado.preferencias[campo] = valor

    def _identificar_item(self, texto):
        if not texto: return None
        t = normalizar(texto).strip()
        m = re.search(r'\b(?:id\s*)?(\d+)\b', t)
        if m and (t.isdigit() or 'id' in t) and 0 <= int(m.group(1)) < len(df):
            return int(m.group(1))
        for i in self.estado.ultimas_recomendacoes:
            if t == normalizar(df.iloc[i]['item']): return i
        for i, nome in enumerate(df['item']):
            if t == normalizar(nome): return i
        return None

    def _recomendar(self, consulta, p):
        parametros = {**self.estado.preferencias, 'consulta': consulta or 'menu',
                      'restricoes_informadas': True, 'intencao':'recomendar'}
        filtrados = filtrar_cardapio(Interpretacao(**parametros))
        grupo = self.estado.categoria_atual
        subtipo = self.estado.subtipo_atual
        if grupo: filtrados = filtrados[filtrados['grupo'] == grupo]
        if subtipo and grupo == 'lanche': filtrados = filtrados[filtrados['subtipo'] == subtipo]
        if filtrados.empty:
            return 'Não encontrei produtos que atendam aos critérios no CSV. Posso mudar a categoria ou flexibilizar os filtros.'
        pares = buscar_similares(consulta or grupo or 'menu', ids_permitidos=filtrados['id_menu'].tolist(), k=len(filtrados))
        pontuacao = dict(pares)
        filtrados = filtrados.copy()
        filtrados['similaridade'] = filtrados['id_menu'].map(pontuacao).fillna(0)
        campos, asc = [], []
        for ativo, campo, crescente in [('prefere_menos_sodio','sodio_mg',True),
                                         ('prefere_menos_calorias','calorias',True),
                                         ('prefere_menos_acucar','acucar_g',True),
                                         ('prefere_mais_proteina','proteina_g',False)]:
            if self.estado.preferencias.get(ativo): campos.append(campo); asc.append(crescente)
        campos += ['similaridade','id_menu']; asc += [False,True]
        itens = filtrados.sort_values(campos, ascending=asc, na_position='last').head(5).to_dict('records')
        self.estado.ultimas_recomendacoes = [int(r['id_menu']) for r in itens]
        aviso = ('\n' + AVISO_ALERGIA) if self.estado.alergias else ''
        return ('Encontrei estas opções do menu.csv:\n' + '\n'.join(formatar_item(r) for r in itens)
                + aviso + '\nQuer adicionar algum ID ao pedido, ver outra categoria ou ajustar suas preferências?')

    def responder(self, mensagem):
        p = interpretar(mensagem)
        t = normalizar(mensagem).strip()
        self.estado.historico.append({'usuario':mensagem,'interpretacao':p.model_dump()})
        if p.intencao == 'reiniciar':
            self.estado = EstadoPedido()
            return 'Pedido reiniciado. Você prefere lanche, sobremesa, bebida, acompanhamento ou café da manhã?'
        if p.intencao == 'ver_pedido':
            if not self.estado.pedido: return 'Seu pedido está vazio. Quer ver lanches, sobremesas ou bebidas?'
            return 'Seu pedido:\n' + '\n'.join(formatar_item(df.iloc[i].to_dict()) for i in self.estado.pedido) + ('\n' + AVISO_ALERGIA if self.estado.alergias else '')
        if p.sem_alergias:
            self.estado.sem_alergias_declaradas = True
            self.estado.alergias = []
            self.estado.ciente_limitacao_alergias = False
        if p.alergias:
            novas = [normalizar(a).strip() for a in p.alergias if str(a).strip()]
            self.estado.alergias = sorted(set(self.estado.alergias + novas))
            self.estado.sem_alergias_declaradas = False
            self.estado.ciente_limitacao_alergias = False
        if p.confirmou_risco_alergia:
            self.estado.ciente_limitacao_alergias = True
        if p.intencao in ('adicionar','remover'):
            id_ = self._identificar_item(p.item_referido or p.consulta or mensagem)
            if id_ is None: return 'Qual produto? Informe o ID exibido ou o nome exato.'
            if p.intencao == 'remover':
                if id_ not in self.estado.pedido: return 'Esse produto não está no seu pedido.'
                self.estado.pedido.remove(id_)
                return f'Removido: {df.iloc[id_]["item"]} (ID {id_}).'
            if self.estado.alergias and not self.estado.ciente_limitacao_alergias:
                return ('Antes de adicionar: você informou alergia a ' + ', '.join(self.estado.alergias)
                        + '. ' + AVISO_ALERGIA + '\nSe quiser apenas registrar o item no pedido, '
                        'diga explicitamente: "Estou ciente de que não há confirmação de segurança; adicionar ID '
                        + str(id_) + '".')
            self.estado.pedido.append(id_)
            return f'Adicionado: {df.iloc[id_]["item"]} (ID {id_}).' + ('\n' + AVISO_ALERGIA if self.estado.alergias else '')
        # Os dados de alergias são persistidos independentemente da categoria.
        if self.estado.alergias and p.intencao == 'pergunta' and any(x in t for x in ['seguro','alerg','contem leite','sem gluten','sem lactose']):
            return AVISO_ALERGIA
        if p.categoria_desejada:
            grupo = extrair_grupo(p.categoria_desejada)
        else:
            grupo = extrair_grupo(mensagem)
        if grupo:
            self.estado.categoria_atual = grupo
            self.estado.subtipo_atual = ''
        subtipo = extrair_subtipo(p.subtipo_lanche or '') or extrair_subtipo(mensagem)
        if subtipo and (grupo in (None, 'lanche') or self.estado.categoria_atual == 'lanche'):
            self.estado.categoria_atual = 'lanche'
            self.estado.subtipo_atual = subtipo
        self._preferencias(p)
        if self.estado.aguardando_restricoes:
            self.estado.aguardando_restricoes = False
            if t in ('nao','nenhuma','nenhum','sem restricoes','sem restricao','nao tenho'):
                p.restricoes_informadas = True
            elif not (p.restricoes_informadas or p.alergias or p.sem_alergias):
                self.estado.aguardando_restricoes = True
                return 'Você tem alguma restrição ou alergia? Responda, por exemplo, "sem restrições e sem alergias".'
        elif not (p.restricoes_informadas or p.alergias or p.sem_alergias or self.estado.preferencias or self.estado.sem_alergias_declaradas or self.estado.alergias):
            self.estado.aguardando_restricoes = True
            self.estado.consulta_pendente = p.consulta or mensagem
            return ('Antes de recomendar, você tem alguma preferência, restrição nutricional ou alergia? '
                    'Pode dizer "sem restrições e sem alergias".')
        if not self.estado.categoria_atual:
            self.estado.aguardando_categoria = True
            return 'Você prefere lanche (frango, bovino, peixe ou nuggets), sobremesa, bebida, acompanhamento ou café da manhã?'
        if self.estado.categoria_atual == 'lanche' and not self.estado.subtipo_atual and any(x in t for x in ['quais tipos','divisoes','tipos de lanche']):
            return 'Divisões de lanches: frango, bovino, peixe, nuggets e outros. Qual você prefere?'
        consulta = p.consulta or self.estado.consulta_pendente or self.estado.categoria_atual
        if t in ('nao','nenhuma','nenhum','sem restricoes','sem restricao','sem restricoes e sem alergias'):
            consulta = self.estado.consulta_pendente or self.estado.categoria_atual
        self.estado.consulta_pendente = ''
        return self._recomendar(consulta, p)

assistente = AssistenteMcDonalds()
print('Assistente inicializado.')

Assistente inicializado.


Interface de conversa no Colab

Execute a célula abaixo e converse em português. Digite sair para encerrar. Exemplos: “quero um lanche”, “frango com menos sódio”, “adicione ID 12”, “mostre meu pedido”.

In [27]:
def conversar():
    print("Assistente McDonald's (baseado no menu.csv). Digite 'sair' para encerrar.")
    while True:
        entrada = input("Você: ").strip()
        if normalizar(entrada) in {"sair", "encerrar", "exit"}:
            break
        if not entrada:
            continue
        try:
            print("Assistente:", assistente.responder(entrada), "\n")
        except Exception as erro:
            print("Falha na consulta ao LLM:", type(erro).__name__, str(erro)[:300])

# Descomente para iniciar a interação (a célula bloqueia enquanto aguarda mensagens):
conversar()

Assistente McDonald's (baseado no menu.csv). Digite 'sair' para encerrar.
Você: quero lanche
Assistente: Antes de recomendar, você tem alguma preferência, restrição nutricional ou alergia? Pode dizer "sem restrições e sem alergias". 

Você: eu nao como carne de vaca
Assistente: Encontrei estas opções do menu.csv:
[ID 42] Big Mac — Beef & Pork; 530 kcal; sódio 960 mg; açúcar 9 g; proteína 24 g
[ID 48] Hamburger — Beef & Pork; 240 kcal; sódio 480 mg; açúcar 6 g; proteína 12 g
[ID 49] Cheeseburger — Beef & Pork; 290 kcal; sódio 680 mg; açúcar 7 g; proteína 15 g
[ID 50] Double Cheeseburger — Beef & Pork; 430 kcal; sódio 1040 mg; açúcar 7 g; proteína 24 g
[ID 51] Bacon Clubhouse Burger — Beef & Pork; 720 kcal; sódio 1470 mg; açúcar 14 g; proteína 39 g
Quer adicionar algum ID ao pedido, ver outra categoria ou ajustar suas preferências? 

Você: eu só como frango
Assistente: Encontrei estas opções do menu.csv:
[ID 57] Premium Crispy Chicken Classic Sandwich — Chicken & Fish; 510 kcal; sódio 99